<a href="https://colab.research.google.com/github/spklapjs/Point-6/blob/main/ai_model/notebooks/03_model_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install protobuf==4.25.5 onnx2tf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.2/223.2 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.

In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torch.onnx
import tensorflow as tf
from google.colab import drive

# 1. 구글 드라이브 마운트
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#경로설정
BASE_DIR = '/content/drive/MyDrive/point-6/ai_model'
CHECKPOINT_DIR = os.path.join(BASE_DIR, 'checkpoints')
EXPORT_DIR = os.path.join(BASE_DIR, 'exported_models')

os.makedirs(EXPORT_DIR, exist_ok=True)

In [ ]:
# 2. CNN-LSTM 하이브리드 모델 구조 정의 및 가중치 로드
class CNNLSTMHybrid(nn.Module):
    def __init__(self, num_classes=6):
        super(CNNLSTMHybrid, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=8, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Conv1d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(64)
        )
        self.lstm = nn.LSTM(input_size=64, hidden_size=128, batch_first=True)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        out = self.cnn(x)
        out = out.permute(0, 2, 1)
        out, _ = self.lstm(out)
        out = self.fc(out[:, -1, :])
        return out

model = CNNLSTMHybrid(num_classes=6)
model.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, 'best_model.pth')))
model.eval()

print("최고 성능 가중치가 적용된 원본 모델 로드 완료")

In [ ]:
# 3. 채널 푸르닝 적용 (목표 희소성 93퍼센트)
for name, module in model.named_modules():
    if isinstance(module, nn.Conv1d):
        # 채널 단위로 L1 노름 기준 93퍼센트 가지치기 수행
        prune.ln_structured(module, name='weight', amount=0.93, n=1, dim=0)
        prune.remove(module, 'weight')

print("채널 푸르닝 적용 완료. 미세 조정(Fine-tuning)을 진행하여 정확도를 86퍼센트 이상으로 복구해야 합니다.")
# 주의: 이 시점에서 02_model_training.ipynb의 학습 루프를 재호출하여 미세 조정을 수행하는 코드가 들어가야 합니다.

In [ ]:
# 4. 푸르닝된 파이토치 모델을 ONNX 형식으로 변환 및 내보내기
dummy_input = torch.randn(1, 8, 20)  # 배치 1, 8축, 200ms 윈도우 사이즈 샘플
onnx_model_path = os.path.join(EXPORT_DIR, 'pruned_model.onnx')

torch.onnx.export(
    model,
    dummy_input,
    onnx_model_path,
    export_params=True,
    opset_version=13,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output']
)
print(f"ONNX 모델 변환 완료: {onnx_model_path}")

In [ ]:
# 5. ONNX 모델을 TensorFlow SavedModel 형식으로 변환 후 TFLite INT8 선형 양자화 적용
print("ONNX 모델을 TensorFlow SavedModel 형식으로 변환합니다...")
os.system(f"onnx2tf -i {onnx_model_path} -o {EXPORT_DIR}/saved_model")

print("INT8 선형 양자화를 적용하여 TFLite 모델로 변환합니다...")
converter = tf.lite.TFLiteConverter.from_saved_model(f"{EXPORT_DIR}/saved_model")
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# 8비트 정수형 선형 양자화를 위한 대표 데이터셋 생성 함수
def representative_dataset():
    for _ in range(100):
        yield [tf.random.normal([3-5], dtype=tf.float32)]

converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_quant_model = converter.convert()

tflite_model_path = os.path.join(EXPORT_DIR, 'motion_model.tflite')
with open(tflite_model_path, 'wb') as f:
    f.write(tflite_quant_model)

print(f"INT8 양자화 TFLite 모델 변환 완료: {tflite_model_path}")